<a href="https://colab.research.google.com/github/StAandrew/listing-parser/blob/main/Listing_Parser_Fine_Tune_Unsloth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Listing description fine-tuning

# Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install "transformers>=4.57,<4.60"
!pip install --no-deps trl==0.22.2
!pip install --quiet wandb weave
!pip install --quiet git+https://github.com/StAandrew/listing-parser.git

# Unsloth

In [4]:
!pip install modelscope
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
import torch
from google.colab import userdata

max_seq_length = 8192 # Choose any! We auto support RoPE Scaling internally!
# 8192 because: our system prompt is ~5.5k tokens (closed-vocab
# reinforcement + 3 few-shot examples from examples.json), plus the
# user message (typical listing 500-2000 tokens) plus the expected
# assistant JSON (400-1200 tokens). 4k would silently truncate long
# rows — the label loss would still compute but on a clipped
# sequence, quietly biasing the fine-tune toward short listings.

dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

import os; os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = userdata.get("HF_TOKEN"),
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 100.1 MB/s eta 0:00:00
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [2]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
import torch
from google.colab import userdata

max_seq_length = 8192 # Choose any! We auto support RoPE Scaling internally!
# 8192 because: our system prompt is ~5.5k tokens (closed-vocab
# reinforcement + 3 few-shot examples from examples.json), plus the
# user message (typical listing 500-2000 tokens) plus the expected
# assistant JSON (400-1200 tokens). 4k would silently truncate long
# rows — the label loss would still compute but on a clipped
# sequence, quietly biasing the fine-tune toward short listings.

dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
# Note: some model names are outdated
fourbit_models = [
    "unsloth/mistral-7b-v0.3-bnb-4bit",      # New Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/llama-3-8b-bnb-4bit",           # Llama-3 15 trillion tokens model 2x faster!
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    "unsloth/llama-3-70b-bnb-4bit",
    "unsloth/Phi-3-mini-4k-instruct",        # Phi-3 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit",             # Gemma 2.2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = userdata.get("HF_TOKEN"),
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


TimeoutError: Unsloth: HuggingFace seems to be down after trying for 120 seconds :(
Check https://status.huggingface.co/ for more details.
As a temporary measure, use modelscope with the same model name ie:
```
pip install modelscope
import os; os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
from unsloth import FastLanguageModel
model = FastLanguageModel.from_pretrained('unsloth/gpt-oss-20b')
```

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    # r=32 (vs tutorial default 16): structured-extraction tasks on
    # closed vocabularies benefit from more adapter capacity than
    # chat-style fine-tunes. Bump to 64 if the loss plateaus above
    # ~0.4; drop to 16 if you're tight on VRAM.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,  # alpha == r is the stable starting point
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Unsloth 2026.4.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Trainable params: 83,886,080 / 4,624,486,400 (1.81%)


In [10]:
# Patch {% generation %} markers into the Llama-3.1 chat template.
# Must run BEFORE any tokenization so assistant_only_loss / the
# AssistantMaskCollator can find assistant spans. See the saga in
# docs/training-notes.md (or our chat history) for why neither the
# Unsloth bundled template nor Meta's own template have these markers
# by default.
LLAMA31_TRAINING_TEMPLATE = """{{- bos_token }}
{%- for message in messages %}
    {%- if message['role'] == 'assistant' %}
        {{- '<|start_header_id|>assistant<|end_header_id|>\\n\\n' }}
        {%- generation %}
        {{- message['content'] | trim + '<|eot_id|>' }}
        {%- endgeneration %}
    {%- else %}
        {{- '<|start_header_id|>' + message['role'] + '<|end_header_id|>\\n\\n' + message['content'] | trim + '<|eot_id|>' }}
    {%- endif %}
{%- endfor %}
{{- '<|start_header_id|>assistant<|end_header_id|>\\n\\n' }}"""
tokenizer.chat_template = LLAMA31_TRAINING_TEMPLATE
print("chat template patched with {% generation %} markers")

chat template patched with {% generation %} markers


# Data prep

In [6]:
from datasets import load_dataset

dataset = load_dataset("standrey/listing-descriptions", split = "train")
print(dataset.column_names)

['description', 'listing_type', 'output']


In [8]:
import urllib.request, os

# prompt.md and examples.json aren't bundled with the pip-installed
# package — they're at the repo root, not inside src/listing_parser/.
# Download them so `build_system_prompt()` finds them via its
# Path.cwd() fallback.
for fname in ("prompt.md", "examples.json"):
    if not os.path.exists(fname):
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/StAandrew/listing-parser/main/src/listing_parser/_assets/{fname}",
            fname,
        )
        print(f"downloaded {fname}")
    else:
        print(f"{fname} already present")

downloaded prompt.md
downloaded examples.json


In [9]:
from datasets import Features, Sequence, Value, load_dataset
from listing_parser.prompting import build_system_prompt, build_user_message

# Pull the same 1,704 rows we pushed to HF via scripts/push_labels_to_hf.py.
# The dataset has three columns: description, listing_type, output.
# `output` is the teacher's fenced JSON (e.g. "```json\n{...}\n```").
dataset = load_dataset("standrey/listing-descriptions", split="train")
print("Columns:", dataset.column_names)
print("Rows:   ", len(dataset))
print("Per-type histogram:",
      {t: sum(1 for r in dataset if r["listing_type"] == t)
       for t in ("Rent", "Sale", "Room")})

# Build the system prompt ONCE — it's ~5.5k deterministic tokens
# rendered from prompt.md + schema vocab + examples.json. Every row
# shares it, which is why Bedrock's prompt caching makes the teacher
# runs effectively free after warm-up.
SYSTEM_PROMPT = build_system_prompt()
print(f"\nSystem prompt length (chars): {len(SYSTEM_PROMPT):,}")

def to_messages(row):
    """Convert one HF row into the {messages: [...]} format SFTTrainer expects.

    We keep the assistant response fenced exactly the way the teacher
    emitted it, because the runner's parse_output_json strips fences at
    inference time. Training the model to emit fences means the fine-
    tune's raw output matches production inference verbatim.
    """
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": build_user_message(
                row["description"], row["listing_type"])},
            {"role": "assistant", "content": row["output"]},
        ]
    }

formatted = dataset.map(to_messages, remove_columns=dataset.column_names)

# Token-length diagnostic — if the p99 is close to max_seq_length we
# are losing training signal on the longest rows. 8192 should be
# comfortable but let's verify.
import numpy as np
lens = [
    len(tokenizer.apply_chat_template(
        r["messages"],
        tokenize=True,
        add_generation_prompt=False,
    ))
    for r in formatted.select(range(min(200, len(formatted))))
]
print(f"\nSample (n={len(lens)}) token-length: "
      f"median={int(np.median(lens))}, p95={int(np.percentile(lens,95))}, "
      f"max={max(lens)}")

# Peek at one fully-rendered example to confirm the template worked.
print("\n----- formatted[0] first 800 chars -----")
print(tokenizer.apply_chat_template(
    formatted[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,
)[:800])
print("... (truncated)")

# Pre-tokenize with the assistant-mask. We do this explicitly rather
# than letting SFTTrainer handle it because (a) Unsloth's patched
# SFTTrainer demands a `formatting_func` when it sees a `messages`
# column with no `text`, and (b) doing it here lets us inspect the
# mask before burning GPU time. The tokenizer's chat_template has
# been patched (in an earlier cell) with {% generation %} markers
# around the assistant turn, which is what makes
# return_assistant_tokens_mask=True actually populate the mask.
def tokenize_row(row):
    enc = tokenizer.apply_chat_template(
        row["messages"],
        tokenize=True,
        return_assistant_tokens_mask=True,
        return_dict=True,
        truncation=True,
        max_length=max_seq_length,
    )
    return {
        "messages":        row["messages"],
        "input_ids":       enc["input_ids"],
        "assistant_masks": enc["assistant_masks"],
    }

# Forcing features explicitly: without this, `datasets.map` infers the
# Arrow schema from the first row and occasionally drops
# `assistant_masks` when the inference picks a type that doesn't round-
# trip (observed with lists of 0/1 ints on some versions).
# features = Features({
#     "input_ids":       Sequence(Value("int32")),
#     "assistant_masks": Sequence(Value("int8")),
# })

tokenized = formatted.map(
    tokenize_row,
    remove_columns=formatted.column_names,
    desc="tokenizing",
)
assert "assistant_masks" in tokenized.column_names, tokenized.column_names
print("columns after tokenize:", tokenized.column_names)

# Manual 95/5 split. We used to call .train_test_split() here but
# that method silently drops the assistant_masks column in some
# datasets versions — .shuffle() + .select() preserves columns
# verbatim and is the same 95/5 numerically. The "real" eval is the
# 60-row gold set scored with the listing-parser scorer; this small
# holdout only gives us a training-time loss curve to watch for
# overfitting.
tokenized = tokenized.shuffle(seed=3407)
n_eval = max(1, int(0.05 * len(tokenized)))
eval_ds  = tokenized.select(range(n_eval))
train_ds = tokenized.select(range(n_eval, len(tokenized)))
print(f"\nTrain: {len(train_ds)} rows, eval: {len(eval_ds)} rows")
print("train_ds columns:", train_ds.column_names)

# Sanity check — the whole point of the template patching + pre-
# tokenize dance is that the mask is non-zero. If it's all zeros the
# loss is computed on nothing and trainer.train() silently no-ops.
example = train_ds[0]
n_assist = sum(example["assistant_masks"])
print(f"\nsanity: {len(example['input_ids'])} total tokens, "
      f"{n_assist} assistant tokens "
      f"({100*n_assist/len(example['input_ids']):.1f}%)")
assert n_assist > 0, "mask is all zeros — generation markers didn't survive"

Columns: ['description', 'listing_type', 'output']
Rows:    1704
Per-type histogram: {'Rent': 983, 'Sale': 205, 'Room': 516}

System prompt length (chars): 19,922


Map:   0%|          | 0/1704 [00:00<?, ? examples/s]


Sample (n=200) token-length: median=5680, p95=6325, max=7037

----- formatted[0] first 800 chars -----
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a UK property listing description parser. You receive one listing
description (free text, often from Rightmove, Zoopla, or SpareRoom) and
return a single JSON object that extracts structured facts about the
property. The JSON MUST conform to the schema below.

General rules:
  - Return JSON only. No prose, no markdown, no code fences.
  - Extract only facts stated or clearly implied by the description. Never
    guess, never invent. When a field is not mentioned, OMIT it from the
    output (do not output null, do not output "" — leave the key out).
  - Booleans mean "the description states this feature is present / true".
    Do NOT output `false` for things that are simply not mentioned.
    Output `false` only when the
... (truncated)


tokenizing:   0%|          | 0/1704 [00:00<?, ? examples/s]

columns after tokenize: ['messages', 'input_ids', 'assistant_masks']

Train: 1619 rows, eval: 85 rows
train_ds columns: ['messages', 'input_ids', 'assistant_masks']

sanity: 5539 total tokens, 272 assistant tokens (4.9%)


In [14]:
# Belt-and-braces: confirm our training-template renders byte-identical
# text to the canonical Meta template. If this ever diffs, train-time
# and inference-time formatting have drifted and the fine-tune will
# silently learn the wrong format.
from transformers import AutoTokenizer
_ref = AutoTokenizer.from_pretrained("NousResearch/Meta-Llama-3.1-8B-Instruct")

sample = train_ds[0]["messages"]
text_ref = _ref.apply_chat_template(sample, tokenize=False, add_generation_prompt=False)
text_new = tokenizer.apply_chat_template(sample, tokenize=False, add_generation_prompt=False)

if text_ref != text_new:
    import difflib
    diff = "\n".join(difflib.unified_diff(
        text_ref.splitlines(), text_new.splitlines(),
        fromfile="reference", tofile="training", lineterm="",
    ))
    raise RuntimeError("training template output drifted from reference:\n" + diff)
print("OK — training template matches reference byte-for-byte")

OK — training template matches reference byte-for-byte


In [15]:
import torch
from dataclasses import dataclass

@dataclass
class AssistantMaskCollator:
    """Pads input_ids and builds labels with -100 outside the assistant
    span. We keep this explicit because TRL's assistant_only_loss path
    is rejected by Unsloth's patched SFTTrainer for non-conversational
    dataset views, and because doing it here means the training signal
    is byte-identical to what we verified in the sanity check."""
    pad_token_id: int

    def __call__(self, examples):
        max_len = max(len(e["input_ids"]) for e in examples)
        input_ids, attn, labels = [], [], []
        for e in examples:
            ids   = e["input_ids"]
            mask  = e["assistant_masks"]
            pad_n = max_len - len(ids)

            input_ids.append(ids + [self.pad_token_id] * pad_n)
            attn.append([1] * len(ids) + [0] * pad_n)

            row_labels = [
                tok if m == 1 else -100
                for tok, m in zip(ids, mask)
            ] + [-100] * pad_n
            labels.append(row_labels)

        return {
            "input_ids":      torch.tensor(input_ids,  dtype=torch.long),
            "attention_mask": torch.tensor(attn,       dtype=torch.long),
            "labels":         torch.tensor(labels,     dtype=torch.long),
        }

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

collator = AssistantMaskCollator(pad_token_id=tokenizer.pad_token_id)

# Train

First, initiate Wandb

In [17]:
import os

# Pull the key from Colab secrets FIRST, before importing wandb.
# Belt-and-braces: also accept a plain env var so the cell works
# outside Colab.
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except (ImportError, Exception):
    pass  # not in Colab, or secret not set

if not os.environ.get("WANDB_API_KEY"):
    raise RuntimeError(
        "WANDB_API_KEY not set. Add it via Colab's 🔑 sidebar "
        "(name: WANDB_API_KEY) or export it as an env var."
    )

import wandb
import weave

# host=None uses the default public cloud. force=True skips the
# "already logged in / not logged in" heuristic that's been flaky
# on Colab runtimes — we just authenticate with the env var.
wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)

RUN_NAME = "llama-3.1-8b-ft-v1-full"
wandb.init(
    project = "listing-parser",
    name    = RUN_NAME,
    group   = "llama-3.1-8b-ft-v1",
    job_type= "train",
    config  = {
        "base_model":    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
        "dataset":       "standrey/listing-descriptions",
        "n_train_rows":  len(train_ds),
        "n_eval_rows":   len(eval_ds),
        "max_seq_length": max_seq_length,
        "lora_r":        32,
        "lora_alpha":    32,
        "learning_rate": 2e-4,
        "batch_size":    1,
        "grad_accum":    8,
        "phase":         "full",
    },
    reinit = True,
)
print(f"W&B run: {wandb.run.url}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: staandrew to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: Initializing weave.
weave: wandb version 0.26.1 is available!  To upgrade, please run:
weave:  $ pip install wandb --upgrade
weave: Logged in as Weights & Biases user: staandrew.
weave: View Weave data at https://wandb.ai/staandrew/listing-parser/weave


W&B run: https://wandb.ai/staandrew/listing-parser/runs/2wd24uqm


Run a sanity check

In [65]:
row = train_ds[0]
enc = tokenizer.apply_chat_template(
    row["messages"],
    tokenize=True,
    return_assistant_tokens_mask=True,
    return_dict=True,
)
ids, mask = enc["input_ids"], enc["assistant_masks"]
n_assistant = sum(mask)
print(f"total tokens: {len(ids)}, assistant tokens: {n_assistant} "
      f"({100*n_assistant/len(ids):.1f}%)")
if n_assistant == 0:
    raise RuntimeError("chat template is missing {% generation %} markers — "
                       "assistant_only_loss will fail")
print("first assistant token:", repr(tokenizer.decode([ids[mask.index(1)]])))

total tokens: 5539, assistant tokens: 272 (4.9%)
first assistant token: '```'


Actual training

In [66]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from unsloth import is_bfloat16_supported
import wandb

# Phase 1 (quick smoke) vs Phase 2 (full run). You'll execute this
# cell twice: first with PHASE="quick", eyeball the loss curve, then
# with PHASE="full". Setting resume_from_checkpoint=True on the second
# pass continues from wherever phase 1 left off — no wasted compute.
PHASE = "full"  # change to "full" for the second pass

# Values for A100
_common = dict(
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 2,         # effective batch = 8
    learning_rate = 2e-4,
    logging_steps = 10,
    save_strategy = "steps",
    save_total_limit = 3,
    eval_strategy = "steps",
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 3407,
    output_dir = "outputs",
    report_to = "wandb",
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    bf16_full_eval = is_bfloat16_supported(),
    assistant_only_loss = False,  # we handle masking in the collator
    packing = False  # packing risks mixing unrelated
                     # (description, output) pairs across
                     # a sequence boundary.
)

if PHASE == "quick":
    training_args = SFTConfig(
        warmup_steps = 20,
        max_steps = 300,
        save_steps = 100,
        eval_steps = 50,
        **_common,
    )
elif PHASE == "full":
    training_args = SFTConfig(
        warmup_steps = 40,
        num_train_epochs = 3,
        save_steps = 50,
        eval_steps = 50,
        save_strategy = "steps",
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,
        save_total_limit = 4,
        **_common,
    )
else:
    raise ValueError(f"unknown PHASE={PHASE!r}")

print("active wandb run:", wandb.run.id if wandb.run else "NONE")
print("active wandb url:", wandb.run.url if wandb.run else "NONE")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    data_collator = collator,
    args = training_args,
)

# Resume from the quick-phase checkpoint if it exists.
import os
resume = os.path.isdir("outputs") and any(
    d.startswith("checkpoint-") for d in os.listdir("outputs")
)
print(f"PHASE={PHASE}, resume={resume}")

trainer_stats = trainer.train(resume_from_checkpoint=False)

active wandb run: 0txdexz1
active wandb url: https://wandb.ai/staandrew/listing-parser/runs/0txdexz1
PHASE=quick, resume=False


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,619 | Num Epochs = 2 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,0.090000,0.084582
100,0.076400,0.080768
150,0.066900,0.073063
200,0.067600,0.066366
250,0.046200,0.065868
300,0.043700,0.065150


# Save merged bf16 weights

In [67]:
# Merge LoRA adapters into the base model, save at bf16. This is the
# artefact Bedrock Custom Model Import ingests — safetensors + config
# + tokenizer files in the HF layout. ~16GB on disk; Colab's /content
# has room.
#
# A100 note: this cell uses ~32GB peak RAM during the merge. Fine on
# both A100-40GB and A100-80GB.
MERGED_DIR  = "merged_bf16"
RUN_SLUG    = "llama-3.1-8b-ft-v1-full-colab"   # matches benchmarks/runs/<slug>/
HF_REPO     = "standrey/listing-parser-llama31-8b-ft-v1-full"
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method = "merged_16bit",   # bf16 on A100
)
# "tokenizer.model not found" is expected for Llama 3.1 — it uses a
# Tiktoken-based BPE tokenizer, not SentencePiece, so there's no
# tokenizer.model file to emit. The tokenizer.json + special_tokens_map
# files that ARE saved are what Bedrock actually reads.
# Push the merged weights to HF. Keeping versioned repo names means
# ft-v1-quick, ft-v1-full, ft-v2, etc. coexist and we can A/B them.
from huggingface_hub import login
import os
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])
else:
    from google.colab import userdata  # secret manager in Colab
    login(token=userdata.get("HF_TOKEN"))

model.push_to_hub_merged(
    HF_REPO,
    tokenizer,
    save_method = "merged_16bit",
    private = False,   # flip to False once you're happy with it
)

print(f"\nMerged weights in {MERGED_DIR}")
print(f"Pushed to: https://huggingface.co/{HF_REPO}")

config.json:   0%|          | 0.00/956 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   3%|3         | 39.9MB / 1.17GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [05:13<00:00, 78.32s/it]


Unsloth: Merge process complete. Saved to `/content/standrey/listing-parser-llama31-8b-ft-v1`

Merged weights in merged_bf16
Pushed to: https://huggingface.co/standrey/listing-parser-llama31-8b-ft-v1


# Score against the frozen gold set

In [68]:
import json, re, urllib.request, time
from pathlib import Path

# Download the frozen gold set from the repo. Pinning to main is OK
# — test_set.jsonl is immutable by policy (CLAUDE.md priority #1).
GOLD_URL = "https://raw.githubusercontent.com/StAandrew/listing-parser/main/benchmarks/test_set.jsonl"
gold_text = urllib.request.urlopen(GOLD_URL).read().decode("utf-8")
gold_rows = [json.loads(line) for line in gold_text.splitlines() if line.strip()]
print(f"Loaded {len(gold_rows)} gold rows")

# Flip into 2x-faster inference mode. Mandatory for Unsloth-wrapped
# models — without this, the model runs in train mode and inference
# is 2-3x slower.
FastLanguageModel.for_inference(model)

# Reuse the repo's parser so the notebook's predictions.jsonl uses
# the exact same fence-stripping rules the scorer expects.
from listing_parser.cleaning import parse_output_json

def predict_one(description: str, listing_type: str) -> tuple[dict | None, str]:
    """Mirrors BedrockLlamaRunner.predict() semantics: greedy decode,
    no schema retry, single parse attempt. We want the honest first-
    pass signal, not a result inflated by retries."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": build_user_message(description, listing_type)},
    ]
    # return_dict=True gives us both input_ids and attention_mask.
    # Passing attention_mask explicitly silences generate()'s warning
    # about pad_token == eos_token and also prevents it from wrongly
    # masking out legitimate <|eot_id|> turn-end tokens in the prompt.
    enc = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            input_ids      = enc["input_ids"],
            attention_mask = enc["attention_mask"],
            max_new_tokens = 1536,
            do_sample      = False,
            use_cache      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )

    # Strip the prompt tokens; we only want what the model generated.
    raw = tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    pred = parse_output_json(raw)
    return pred, raw

# Write directly into benchmarks/runs/<slug>/ so the path matches
# what `lp-benchmark score` expects locally — no post-download moving.
# Run all 60 rows. T4 @ 8B bf16 gives ~30-60s/row → ~45min total.
# A100 is ~2-3x faster. Progress prints because Colab's buffering
# can make long cells look dead.
# A100 @ 8B bf16 gives ~10-15s/row → ~15min total for 60 rows.

# run_dir = Path("benchmarks/runs") / RUN_SLUG
# run_dir.mkdir(parents=True, exist_ok=True)
# out_path = run_dir / "predictions.jsonl"

run_dir = Path("benchmarks/runs") / RUN_SLUG
run_dir.mkdir(parents=True, exist_ok=True)
out_path = run_dir / "predictions.jsonl"

with out_path.open("w", encoding="utf-8") as f:
    t0 = time.monotonic()
    for i, row in enumerate(gold_rows):
        pred, raw = predict_one(row["description"], row["listing_type"])
        f.write(json.dumps({
            "row_index": row["row_index"],
            "pred": pred,
            "raw": raw,
        }, ensure_ascii=False) + "\n")
        f.flush()
        if (i + 1) % 5 == 0:
            rate = (i + 1) / (time.monotonic() - t0)
            eta = (len(gold_rows) - i - 1) / rate
            print(f"  {i+1}/{len(gold_rows)}  rate={rate:.2f}/s  eta={eta/60:.1f}min")

print(f"\nWrote {out_path} ({out_path.stat().st_size:,} bytes)")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Loaded 60 gold rows
  5/60  rate=0.04/s  eta=24.3min
  10/60  rate=0.04/s  eta=18.9min
  15/60  rate=0.04/s  eta=19.1min
  20/60  rate=0.04/s  eta=15.1min
  25/60  rate=0.04/s  eta=13.3min
  30/60  rate=0.04/s  eta=11.5min
  35/60  rate=0.04/s  eta=9.5min
  40/60  rate=0.04/s  eta=8.0min
  45/60  rate=0.04/s  eta=6.0min
  50/60  rate=0.04/s  eta=3.9min
  55/60  rate=0.04/s  eta=1.9min
  60/60  rate=0.04/s  eta=0.0min

Wrote predictions.jsonl (156,414 bytes)


# Download predictions for local scoring

In [70]:
from google.colab import files

# Download into the same slug-named subdir when you extract locally.
# The exact relative path below matches what `lp-benchmark score`
# expects (benchmarks/runs/<slug>/predictions.jsonl).
files.download(str(out_path))
print(f"\nOn your laptop, move the file into:\n"
      f"  benchmarks/runs/{RUN_SLUG}/predictions.jsonl\n"
      f"then score locally with the CLI below.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

NameError: name 'RUN_SLUG' is not defined

# Score locally


After downloading `predictions.jsonl`, move it into
`benchmarks/runs/llama-3.1-8b-ft-v1-full-colab/` on your laptop and run:

```bash
lp-benchmark score \
    --gold benchmarks/test_set.jsonl \
    --predictions benchmarks/runs/llama-3.1-8b-ft-v1-ful-colab/predictions.jsonl \
    --out-dir benchmarks/runs/llama-3.1-8b-ft-v1-full-colab \
    --name "llama-3.1-8b ft-v1 full (Colab)"
```

Then compare `report.md` against `benchmarks/runs/llama-3.1-8b-base/report.md`
and `benchmarks/runs/haiku-4.5-teacher/report.md` to see where the
fine-tune landed.




# Export to GGUF for Ollama

In [71]:
# Gate this — GGUF conversion is slow (5-15min) and is not what
# Bedrock needs. Set to True only if you want a laptop Ollama copy.
EXPORT_GGUF = False

if EXPORT_GGUF:
    model.save_pretrained_gguf(
        "gguf_q4_k_m",
        tokenizer,
        quantization_method = "q4_k_m",  # same quant level the user's
                                         # local Ollama baseline uses
    )
    # Optional HF push of the GGUF — handy for `ollama pull hf.co/...`:
    model.push_to_hub_gguf(
        "standrey/listing-parser-llama31-8b-ft-v1-gguf",
        tokenizer,
        quantization_method = "q4_k_m",
        private = True,
    )
else:
    print("GGUF export skipped. Set EXPORT_GGUF=True to enable.")

GGUF export skipped. Set EXPORT_GGUF=True to enable.
